**Geochemistry Biplot App for Bruker Results.csv Files**

Jupyter Notebook Version
N. Tripcevich 2026, CC BY-SA 4.0  
[More Information Online](https://github.com/arf-berkeley/bruker-xrf-ppm-plot)

Click here, then proceed through this notebook cell-by-cell by pressing Shift-Return on your keyboard. Follow the instructions provided to upload your .csv file and view the data.

In [ ]:
%%capture
%pip install plotly ipywidgets
import sys
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.colors import DEFAULT_PLOTLY_COLORS
from IPython.display import display
import ipywidgets as widgets
import io

**Select the Results.csv table from your Bruker analysis**

Browse to a copy of the __Results.csv__ file typically found in Bruker/Data/Results.csv

The code below imports Method Weight % results from the most recent analysis from the end of the Results.csv file.

In [ ]:
# Cell 2 - CSV Import - supports both local and hosted environments
from io import StringIO

def is_local():
    """Detect if we're running locally or on a hosted kernel"""
    try:
        import tkinter as tk
        root = tk.Tk()
        root.destroy()
        return True
    except Exception:
        return False

def parse_from_last_header(content_str):
    """Parse CSV from the last Method header downwards.
    Bruker XRF instruments repeat the header row at the start of each
    Method block - we only want data from the most recent one."""
    lines = content_str.splitlines(keepends=True)
    header_line = None
    for i in range(len(lines) - 1, -1, -1):
        if lines[i].split(',')[0].strip() == 'File #':
            header_line = i
            break
    if header_line is None:
        raise ValueError('Could not find header row in file')
    data = ''.join(lines[header_line:])
    df = pd.read_csv(StringIO(data))
    print(f'✓ Found last Method header at line {header_line}')
    print(f'✓ Loaded {len(lines) - header_line - 1} rows from last Method used')
    return df

if is_local():
    import tkinter as tk
    from tkinter import filedialog

    def browse_csv(label):
        root = tk.Tk()
        root.withdraw()
        root.attributes('-topmost', True)
        file_path = filedialog.askopenfilename(
            title=f'Select {label} CSV',
            filetypes=[('CSV files', '*.csv'), ('All files', '*.*')]
        )
        root.destroy()
        return file_path

    study_path = browse_csv('Results.csv')
    if study_path:
        with open(study_path, 'r') as f:
            study_import = parse_from_last_header(f.read())
        print(f'Results.csv loaded: {study_import.shape}')
    else:
        study_import = None

else:
    study_import = None

    study_upload = widgets.FileUpload(accept='.csv', multiple=False)

    confirm_btn = widgets.Button(
        description='Load Data',
        button_style='success',
        icon='check',
        disabled=True
    )
    status_label = widgets.Label('Please Upload a Bruker XRF Results.csv file')

    def update_button(change):
        if study_upload.value:
            confirm_btn.disabled = False
            status_label.value = 'Results.csv file ready - click Load Data to continue'
        else:
            confirm_btn.disabled = True
            status_label.value = 'Please upload a Bruker XRFResults.csv file'

    def load_files(btn):
        global study_import
        try:
            study_content = study_upload.value[0]['content'].tobytes().decode('utf-8')
            study_import = parse_from_last_header(study_content)
            status_label.value = f'✓ Results.csv {study_import.shape} loaded successfully'
            confirm_btn.disabled = True
            confirm_btn.description = 'Loaded'

        except Exception as e:
            status_label.value = f'Error loading files: {e}'

    study_upload.observe(update_button, names='value')
    confirm_btn.on_click(load_files)

    display(widgets.VBox([
        widgets.Label('Upload Results.csv file:'),
        study_upload,
        confirm_btn,
        status_label
    ]))

***[Click Here to Continue]***

__Clean up Bruker data__

Cleaning data includes removing the following: elemental error columns, Alloy, Match Qual columns, Multiplier, Cal Check, Operator, Field 1&2. This script also replaces Below Detection Limits LOD with 0.

In [ ]:
# Cell 3 - Data cleaning
if study_import is None:
    print('Please upload Results.csv before continuing')
else:
    # Metadata columns to keep
    META_COLS = ['File #', 'DateTime', 'Name', 'Application', 'Method', 'ElapsedTime']

    # Non-element columns to drop
    NON_ELEMENT_COLS = ['Alloy 1', 'Match Qual 1', 'Alloy 2', 'Match Qual 2', 
                        'Alloy 3', 'Match Qual 3', 'Multiplier', 'Cal Check',
                        'Operator', 'Field1', 'Field2']

    # Drop error columns and non-element columns
    drop_cols = [c for c in study_import.columns if 'Err' in c or c in NON_ELEMENT_COLS]
    keep_cols = [c for c in study_import.columns if c not in drop_cols]

    study = study_import[keep_cols].copy()

    # Replace below detection limit values with zero
    study = study.replace('< LOD', 0)

    # Ensure correct types
    string_cols = [c for c in ['Name', 'Application', 'Method'] if c in study.columns]
    numeric_cols = [c for c in study.columns if c not in string_cols + ['DateTime']]
    study[string_cols] = study[string_cols].astype('string')
    study[numeric_cols] = study[numeric_cols].apply(pd.to_numeric, errors='coerce')
    # Keep DateTime as datetime
    study['DateTime'] = pd.to_datetime(study['DateTime'])

    # Identify element columns (numeric cols excluding metadata)
    element_cols = [c for c in numeric_cols if c not in ['File #', 'ElapsedTime']]

    # Convert element values from percent to PPM and round to 1 decimal place
    study[element_cols] = (study[element_cols] * 10000).round(1)

    # Drop rows and columns with all NaN
    study.dropna(axis=1, how='all', inplace=True)
    study.dropna(how='all', inplace=True)

    print('Dataset Headers')
    print('Study: ' + str(study.columns.tolist()))
    print(f'Rows after dropping NaN: {len(study)}')

**Display Data Table**

Run the next cell to view the data table before viewing a biplot.

In [ ]:
# Cell 5 - Data Table
if study is None:
    print('Please complete data cleaning before continuing')
else:
    # Format DateTime for display
    display_df = study.copy()
    display_df['DateTime'] = display_df['DateTime'].dt.strftime('%m/%d/%Y %H:%M')
    
    # Identify element columns
    non_element = ['File #', 'DateTime', 'Name', 'ID']
    element_cols = [c for c in display_df.columns if c not in non_element]
    text_cols = [c for c in ['DateTime', 'Name', 'ID'] if c in display_df.columns]
    
    display(display_df.style
        .format({col: '{:.1f}' for col in element_cols})
        .set_properties(**{
            'text-align': 'right',
            'font-size': '12px'
        })
        .set_properties(subset=text_cols, **{
            'text-align': 'left'
        })
        .set_table_styles([{
            'selector': 'th',
            'props': [('text-align', 'center'), ('font-weight', 'bold')]
        }])
        .hide(axis='index')
    )

In [ ]:
# Cell 4 - Biplot
if study is None:
    print('Please complete data cleaning before continuing')
else:
    # Derive element columns by excluding known non-element columns
    NON_ELEMENT_COLS = ['File #', 'DateTime', 'Name', 'ID']
    elements_present = [c for c in study.columns if c not in NON_ELEMENT_COLS]

    x_dropdown = widgets.Dropdown(
        options=elements_present,
        value='Sr' if 'Sr' in elements_present else elements_present[0],
        description='X Axis:'
    )
    y_dropdown = widgets.Dropdown(
        options=elements_present,
        value='Rb' if 'Rb' in elements_present else elements_present[1],
        description='Y Axis:'
    )

    plot_output = widgets.Output()

    def update_plot(change):
        with plot_output:
            plot_output.clear_output(wait=True)
            x = x_dropdown.value
            y = y_dropdown.value
            fig = px.scatter(
                study,
                x=x,
                y=y,
                color='Name',
                hover_data=['File #', 'Name', 'DateTime', x, y],
                title=f'{y} vs {x} Biplot',
                labels={x: f'{x} (PPM)', y: f'{y} (PPM)'}
            )
            fig.update_traces(marker=dict(size=8))
            fig.update_layout(height=600, hovermode='closest')
            fig.show()

    x_dropdown.observe(update_plot, names='value')
    y_dropdown.observe(update_plot, names='value')

    display(widgets.VBox([
        widgets.HBox([x_dropdown, y_dropdown]),
        plot_output
    ]))

    update_plot(None)

In [ ]:
# Cell 5 - Option to export dataset as CSV
from IPython.display import display, HTML
if study is None:
    print('Please complete data cleaning before continuing')
else:
    export_output = widgets.Output()

    export_btn = widgets.Button(
        description='Export CSV',
        button_style='success',
        icon='download'
    )

    def on_export(btn):
        with export_output:
            export_output.clear_output(wait=True)
            if is_local():
                import tkinter as tk
                from tkinter import filedialog

                root = tk.Tk()
                root.withdraw()
                root.attributes('-topmost', True)
                save_path = filedialog.asksaveasfilename(
                    title='Save CSV',
                    defaultextension='.csv',
                    filetypes=[('CSV files', '*.csv'), ('All files', '*.*')],
                    initialfile=f'Bruker_Results_export_{study["DateTime"].max().strftime("%Y%m%d")}.csv'
                )
                root.destroy()

                if save_path:
                    study.to_csv(save_path, index=False)
                    print(f'✓ Exported {len(study)} rows to {save_path}')
                else:
                    print('Export cancelled')
            else:
                import base64
                from IPython.display import HTML
                csv_str = study.to_csv(index=False)
                b64 = base64.b64encode(csv_str.encode()).decode()
                filename = f'Bruker_Results_export_{study["DateTime"].max().strftime("%Y%m%d")}.csv'
                html = f'<a download="{filename}" href="data:text/csv;base64,{b64}">Click here to download {filename}</a>'
                display(HTML(html))

    export_btn.on_click(on_export)

    display(widgets.HTML('<b>Would you like to export the cleaned up CSV of these values?</b>'))
    display(widgets.VBox([export_btn, export_output]))